# Notebook 09 — Deploy a Genie-Powered Databricks App

**What you’ll learn:** How to wrap your Genie space in a custom web application that anyone can bookmark and use — no Databricks workspace login required.

**Why an app?** Genie spaces live inside the Databricks workspace UI. A Databricks App gives you a standalone URL with your own branding, starter questions, and chat interface. The app calls the Genie Conversation API behind the scenes, so Unity Catalog governance still applies to every query.

**What this notebook does:**

| Step | What happens |
|------|--------------|
| **1** | Load your Genie Space ID from `workshop_config` |
| **2** | Generate app code (Flask + HTML/CSS/JS) and upload to workspace |
| **3** | Create & deploy the app, grant permissions to its service principal |
| **4** | Click the live app link and ask questions |

**Before you start:** Run notebook **03** (Genie spaces) so `genie_space_id` exists in `workshop_config`.

**Compute:** Serverless.

## Compute

Use **Serverless** for this notebook. The deployed app uses its own serverless compute.

## Step 1 — Load Genie Space ID

Reads the space ID saved by notebook **03**. All subsequent cells use this ID.

In [ ]:
%run ./00_workshop_config

In [ ]:
import re
import html as _html
import requests
import base64
import time
import json
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host_url = w.config.host.rstrip("/")

def genie_ui_room_url(space_id):
    m = re.search(r"adb-(\d+)\.", host_url)
    o = m.group(1) if m else ""
    q = f"?o={o}" if o else ""
    return f"{host_url}/genie/rooms/{space_id}{q}"

# Load the space IDs saved by notebook 03 so we know which Genie space to connect the app to
_cfg = spark.table(full_table("workshop_config")).toPandas().set_index("key")["value"].to_dict()
GENIE_SPACE_ID = _cfg.get("genie_space_id", "")
if not GENIE_SPACE_ID:
    raise RuntimeError("genie_space_id not found in workshop_config. Run notebook 03 first.")

print(f"Genie Space ID : {GENIE_SPACE_ID}")
print(f"Space URL      : {genie_ui_room_url(GENIE_SPACE_ID)}")


## Step 2 — Upload app code to workspace

Generates three files (`app.py`, `app.yaml`, `requirements.txt`) with the Genie Space ID injected, and uploads them to a workspace folder next to this notebook.

The app is a **Flask** server with a branded chat UI:
- Dark Databricks-style header with manufacturing branding
- Starter question buttons for common manufacturing queries
- Chat interface powered by the Genie Conversation API
- Unity Catalog governance enforced on every query

In [ ]:
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
token = ctx.apiToken().get()
user_email = ctx.userName().get()
api_auth = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# The app code will be uploaded to a workspace folder under your user directory
app_path = f"/Workspace/Users/{user_email}/manufacturing-genie-app"
requests.post(f"{host_url}/api/2.0/workspace/mkdirs", headers=api_auth, json={"path": app_path})

# ── app.yaml ────────────────────────────────────────────────
app_yaml = f"""command:
  - python
  - app.py

env:
  - name: GENIE_SPACE_ID
    value: "{GENIE_SPACE_ID}"

resources:
  - name: sql-warehouse
    sql_warehouse:
      permission: CAN_USE
"""

# ── requirements.txt ─────────────────────────────────────────
requirements = "flask>=3.0.0\ndatabricks-sdk>=0.30.0\n"

# ── app.py ───────────────────────────────────────────────────
app_py = r'''
import os, requests, time, json
from flask import Flask, request, jsonify, render_template_string
from databricks.sdk import WorkspaceClient

SID = os.environ.get("GENIE_SPACE_ID", "")
w = WorkspaceClient()
H = w.config.host.rstrip("/")
app = Flask(__name__)

HTML = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1">
  <title>Manufacturing Quality Analyzer</title>
  <style>
    :root { --dark:#1B3139; --red:#FF3621; --green:#00A972; --amber:#E8A317; --bg:#0f1923; --card:#1a2d38; --border:#2a4a5a; --text:#e2e8f0; --muted:#8899aa; }
    * { margin:0; padding:0; box-sizing:border-box; }
    body { font-family:system-ui,sans-serif; background:var(--bg); color:var(--text); height:100vh; display:flex; flex-direction:column; }
    .header { background:linear-gradient(135deg,var(--dark),#0d1b24); padding:20px 28px; border-bottom:3px solid var(--red); display:flex; justify-content:space-between; align-items:center; }
    .header h1 { font-size:22px; font-weight:700; } .header h1 span { color:var(--red); }
    .header p { color:var(--muted); font-size:12px; margin-top:4px; }
    .badge { background:var(--red); color:white; padding:4px 12px; border-radius:20px; font-size:11px; font-weight:600; }
    .main { flex:1; display:flex; overflow:hidden; }
    .sidebar { width:240px; background:#111d27; border-right:1px solid var(--border); padding:16px; overflow-y:auto; }
    .sidebar h3 { font-size:12px; color:var(--muted); text-transform:uppercase; letter-spacing:1px; margin-bottom:12px; }
    .starter { display:block; width:100%%%%; text-align:left; padding:10px 12px; background:var(--card); border:1px solid var(--border); border-radius:8px; color:var(--text); cursor:pointer; font-size:12px; margin-bottom:8px; transition:all 0.2s; }
    .starter:hover { border-color:var(--red); background:#1e3545; transform:translateX(4px); }
    .chat-area { flex:1; display:flex; flex-direction:column; }
    .chat { flex:1; overflow-y:auto; padding:20px 28px; }
    .msg { max-width:85%%%%; margin:12px 0; animation:fadeIn 0.3s ease; }
    @keyframes fadeIn { from{opacity:0;transform:translateY(8px)} to{opacity:1;transform:none} }
    .msg-user { margin-left:auto; }
    .msg-label { font-size:10px; color:var(--muted); margin-bottom:4px; text-transform:uppercase; }
    .msg-user .msg-label { text-align:right; }
    .msg-bubble { padding:14px 18px; border-radius:12px; line-height:1.6; font-size:14px; }
    .msg-user .msg-bubble { background:linear-gradient(135deg,var(--red),#c4280f); color:white; border-bottom-right-radius:4px; }
    .msg-bot .msg-bubble { background:var(--card); border:1px solid var(--border); border-bottom-left-radius:4px; }
    .msg-bot table { border-collapse:collapse; width:100%%%%; margin:10px 0; font-size:12px; }
    .msg-bot th { background:var(--dark); color:white; padding:8px 10px; text-align:left; font-size:11px; text-transform:uppercase; }
    .msg-bot td { padding:8px 10px; border-bottom:1px solid var(--border); }
    .msg-bot tr:hover { background:#1e3545; }
    .typing { color:var(--muted); font-style:italic; }
    .typing::after { content:''; animation:dots 1.5s infinite; }
    @keyframes dots { 0%%%%{content:'.'} 33%%%%{content:'..'} 66%%%%{content:'...'} }
    .input-bar { display:flex; padding:16px 28px; background:#111d27; border-top:1px solid var(--border); gap:10px; }
    .input-bar input { flex:1; padding:14px 18px; background:var(--card); border:1px solid var(--border); border-radius:10px; color:var(--text); font-size:14px; outline:none; }
    .input-bar input:focus { border-color:var(--red); }
    .input-bar button { padding:14px 28px; background:var(--red); color:white; border:none; border-radius:10px; cursor:pointer; font-size:14px; font-weight:600; }
    .input-bar button:hover { background:#e02d1a; }
    .footer { text-align:center; padding:8px; font-size:10px; color:var(--muted); border-top:1px solid var(--border); }
    .welcome { text-align:center; padding:60px 40px; color:var(--muted); }
    .welcome h2 { font-size:28px; color:var(--text); margin-bottom:12px; }
  </style>
</head>
<body>
  <div class="header"><div><h1><span>Manufacturing</span> Quality Analyzer</h1><p>Powered by Databricks Genie</p></div><span class="badge">LIVE</span></div>
  <div class="main">
    <div class="sidebar">
      <h3>Quick Questions</h3>
      <button class="starter" onclick="ask(this.innerText)">Average OEE by plant for 2024?</button>
      <button class="starter" onclick="ask(this.innerText)">Which lines had the most downtime?</button>
      <button class="starter" onclick="ask(this.innerText)">Top 5 defect codes this year?</button>
      <button class="starter" onclick="ask(this.innerText)">Safety incidents by severity?</button>
      <button class="starter" onclick="ask(this.innerText)">First pass yield trend by month?</button>
      <button class="starter" onclick="ask(this.innerText)">Plants with most rework events?</button>
    </div>
    <div class="chat-area">
      <div class="chat" id="chat">
        <div class="welcome" id="welcome"><h2>Ask anything about manufacturing quality</h2><p>Click a quick question or type your own below.</p></div>
      </div>
      <div class="input-bar">
        <input id="q" placeholder="Ask about OEE, defects, downtime, safety..." onkeydown="if(event.key==='Enter')ask()">
        <button onclick="ask()">Ask Genie</button>
      </div>
    </div>
  </div>
  <div class="footer">Powered by Databricks Genie &bull; Unity Catalog Governed</div>
  <script>
    function ask(text){const q=text||document.getElementById('q').value;if(!q.trim())return;const c=document.getElementById('chat');const w=document.getElementById('welcome');if(w)w.remove();c.innerHTML+='<div class="msg msg-user"><div class="msg-label">You</div><div class="msg-bubble">'+q+'</div></div>';c.innerHTML+='<div class="msg msg-bot" id="typing"><div class="msg-label">Genie</div><div class="msg-bubble typing">Analyzing</div></div>';document.getElementById('q').value='';c.scrollTop=c.scrollHeight;fetch('/ask',{method:'POST',headers:{'Content-Type':'application/json'},body:JSON.stringify({question:q})}).then(r=>r.json()).then(d=>{document.getElementById('typing').remove();c.innerHTML+='<div class="msg msg-bot"><div class="msg-label">Genie</div><div class="msg-bubble">'+d.answer+'</div></div>';c.scrollTop=c.scrollHeight;}).catch(e=>{document.getElementById('typing').remove();c.innerHTML+='<div class="msg msg-bot"><div class="msg-label">Genie</div><div class="msg-bubble">Error: '+e+'</div></div>';});}
  </script>
</body></html>
"""

@app.route("/")
def index(): return render_template_string(HTML)

@app.route("/ask", methods=["POST"])
def ask_genie():
    q = request.json.get("question", "")
    try:
        auth = {**w.config.authenticate(), "Content-Type":"application/json"}
        resp = requests.post(f"{H}/api/2.0/genie/spaces/{SID}/start-conversation", headers=auth, json={"content":q})
        if resp.status_code != 200:
            return jsonify({"answer": f"API error {resp.status_code}: {resp.text[:200]}"})
        s = resp.json()
        cid, mid = s.get("conversation_id"), s.get("message_id")
        for _ in range(20):
            time.sleep(3)
            p = requests.get(f"{H}/api/2.0/genie/spaces/{SID}/conversations/{cid}/messages/{mid}", headers=auth).json()
            if p.get("status") == "COMPLETED":
                for a in p.get("attachments", []):
                    if a.get("query"):
                        aid = a.get("attachment_id") or a.get("id", "")
                        qr = requests.get(f"{H}/api/2.0/genie/spaces/{SID}/conversations/{cid}/messages/{mid}/query-result/{aid}", headers=auth).json()
                        data = qr.get("statement_response", {}).get("result", {})
                        cols = [c["name"] for c in data.get("schema", {}).get("columns", [])]
                        rows = data.get("data_array", [])[:25]
                        t = "<table><tr>" + "".join(f"<th>{c}</th>" for c in cols) + "</tr>"
                        for r in rows:
                            t += "<tr>" + "".join(f"<td>{v if v else ''}</td>" for v in r) + "</tr>"
                        t += f"</table><div style='margin-top:8px;font-size:11px;color:#8899aa'>{len(rows)} rows</div>"
                        return jsonify({"answer": t})
                return jsonify({"answer": p.get("content", "Done.")})
            elif p.get("status") in ["FAILED", "CANCELLED"]:
                return jsonify({"answer": f"Genie {p.get('status')}"})
        return jsonify({"answer": "Timeout."})
    except Exception as e:
        return jsonify({"answer": f"Error: {str(e)}"})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=8000)
'''

# Upload files
for filename, content in [('app.py', app_py), ('app.yaml', app_yaml), ('requirements.txt', requirements)]:
    encoded = base64.b64encode(content.encode()).decode()
    resp = requests.post(
        f"{host_url}/api/2.0/workspace/import",
        headers=api_auth,
        json={"path": f"{app_path}/{filename}", "content": encoded, "overwrite": True, "format": "AUTO"},
    )
    status = 'OK' if resp.status_code == 200 else f'FAIL ({resp.status_code})'
    print(f"  {filename}: {status}")

print(f"\nApp uploaded to: {app_path}")


## Step 3 — Deploy the app and grant permissions

Creates a Databricks App, deploys from the workspace folder, discovers the app’s service principal, and grants it access to your catalog/schema and Genie space.

When complete, a **clickable app link** appears below.

In [ ]:
APP_NAME = "manufacturing-genie-app"

# Step A: Create the app (or reuse if it exists)
print(f"Creating app: {APP_NAME}...")
cr = requests.post(
    f"{host_url}/api/2.0/apps",
    headers=api_auth,
    json={"name": APP_NAME, "description": "Manufacturing Quality Analyzer (Genie)"},
)
if cr.status_code in (200, 201):
    print("App created.")
elif cr.status_code == 409:
    print("App already exists — reusing.")
else:
    print(f"Create returned {cr.status_code}: {cr.text[:200]}")

# Step B: Wait for the app to reach a deployable state before pushing code
print("Waiting for app to be ready for deployment...")
for _wait in range(24):
    time.sleep(5)
    _sr = requests.get(f"{host_url}/api/2.0/apps/{APP_NAME}", headers=api_auth)
    if _sr.status_code == 200:
        _state = _sr.json().get("status", {}).get("state", _sr.json().get("state", ""))
        if _state in ("RUNNING", "IDLE", "READY", "DEPLOYED", "STOPPED"):
            print(f"App state: {_state} — ready to deploy.")
            break
        elif _state in ("FAILED", "ERROR"):
            print(f"App in {_state} state — attempting deploy anyway.")
            break
else:
    print("Timed out waiting. Attempting deploy anyway.")

# Step C: Deploy the app code
print(f"Deploying from {app_path}...")
dr = requests.post(
    f"{host_url}/api/2.0/apps/{APP_NAME}/deployments",
    headers=api_auth,
    json={"source_code_path": app_path},
)
if dr.status_code not in (200, 201):
    dr = requests.post(
        f"{host_url}/api/2.0/apps/{APP_NAME}/deployments",
        headers=api_auth,
        json={"deployment_artifacts": {"source_code_path": app_path}},
    )

app_url = ""
if dr.status_code in (200, 201):
    print("Deployment started. Waiting for app to become live...")
    for _ in range(30):
        time.sleep(5)
        sr = requests.get(f"{host_url}/api/2.0/apps/{APP_NAME}", headers=api_auth)
        if sr.status_code == 200:
            d = sr.json()
            state = d.get("status", {}).get("state", d.get("state", ""))
            app_url = d.get("url", "")
            if state in ("RUNNING", "READY", "DEPLOYED"):
                print(f"App is live: {app_url}")
                break
            elif state in ("FAILED", "ERROR"):
                print(f"Deploy {state}: {d.get('status', {}).get('message', '')}")
                break
    else:
        print(f"Still deploying. Check: {host_url}/compute/apps")
else:
    print(f"Deploy call failed ({dr.status_code}): {dr.text[:200]}")

# Step D: Discover the app's service principal and grant permissions
app_info = requests.get(f"{host_url}/api/2.0/apps/{APP_NAME}", headers=api_auth).json()
sp_name = (app_info.get("service_principal_name")
           or app_info.get("effective_service_principal_name", ""))
sp_id = (app_info.get("service_principal_id")
         or app_info.get("effective_service_principal_id", ""))

# SCIM fallback: if sp_name is empty, look up the SP by id
if not sp_name and sp_id:
    scim = requests.get(
        f"{host_url}/api/2.0/preview/scim/v2/ServicePrincipals?count=200",
        headers=api_auth,
    )
    if scim.status_code == 200:
        for _sp in scim.json().get("Resources", []):
            if _sp.get("id") == str(sp_id):
                sp_name = _sp.get("displayName", "")
                break

if sp_name:
    print(f"\nGranting permissions to SP: {sp_name}")

    # UC catalog/schema grants
    for sql in [
        f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{sp_name}`",
        f"GRANT USE SCHEMA ON SCHEMA {CATALOG}.{SCHEMA} TO `{sp_name}`",
        f"GRANT SELECT ON SCHEMA {CATALOG}.{SCHEMA} TO `{sp_name}`",
    ]:
        try:
            spark.sql(sql)
            print(f"  OK: {sql.split('GRANT ')[1].split(' TO')[0]}")
        except Exception as e:
            print(f"  WARN: {str(e)[:80]}")

    # Genie space permissions (try multiple endpoints)
    for endpoint in [
        f"{host_url}/api/2.0/permissions/data-rooms/{GENIE_SPACE_ID}",
        f"{host_url}/api/2.0/permissions/dashboards/{GENIE_SPACE_ID}",
    ]:
        pr = requests.patch(endpoint, headers=api_auth, json={
            "access_control_list": [
                {"service_principal_name": sp_name, "permission_level": "CAN_RUN"},
                {"group_name": "users", "permission_level": "CAN_RUN"},
            ]
        })
        if pr.status_code == 200:
            print("  OK: Genie space permissions")
            break

    # SQL warehouse access
    for wh in list(w.warehouses.list()):
        wr = requests.patch(
            f"{host_url}/api/2.0/permissions/sql/warehouses/{wh.id}",
            headers=api_auth,
            json={"access_control_list": [{"service_principal_name": sp_name, "permission_level": "CAN_USE"}]},
        )
        if wr.status_code == 200:
            print(f"  OK: SQL warehouse access ({wh.name})")
            break
else:
    print("Could not find app service principal. Grant permissions manually:")
    print(f"  Genie Space -> Share -> Add the SP with CAN USE")
    print(f"  GRANT SELECT ON SCHEMA {CATALOG}.{SCHEMA} TO `<sp-name>`")

if app_url:
    displayHTML(
        f'<div style="margin:16px 0">'
        f'<a href="{_html.escape(app_url, quote=True)}" target="_blank" '
        f'style="font-size:16px;padding:14px 28px;background:#FF3621;color:white;border-radius:8px;'
        f'text-decoration:none;font-weight:600;">Open Manufacturing Genie App</a></div>'
    )


## Step 4 — Try it

Click the app link above. Try these questions:

- *Average OEE by plant for 2024?*
- *Which lines had the most downtime?*
- *Top 5 defect codes this year?*
- *Safety incidents by severity?*
- *First pass yield trend by month?*

If you see `PERMISSION_DENIED`, re-run Step 3 or grant manually:
- **Genie space:** Share → add the app’s service principal with CAN USE
- **Catalog:** `GRANT SELECT ON SCHEMA {catalog}.{schema} TO \`sp-name\``

## Done

Your manufacturing Genie app is deployed. Anyone with the URL can ask questions in plain English, governed by Unity Catalog.

**Next:** Notebook **10** (monitoring) or **11** (CI/CD for Genie spaces).